# Multi-Task Neurologist Specialist GPU Training
### MIMIC-IV Clinical + Notes + ED Unified Dataset (67,028 Encounters)

This notebook fine-tunes **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`) for multi-task clinical neurology inference:
1. **Neurological Disease Identification** (6 multi-label classes: `stroke_ischemic`, `intracranial_hemorrhage`, `epilepsy_seizures`, `altered_mental_status_encephalopathy`, `transient_ischemic_attack`, `neuropathy_neurodegenerative`)
2. **Chief Complaint Triage** (11 single-label classes: `altered_mental_status`, `confusion_memory`, `dizziness_vertigo`, `headache_migraine`, `loss_of_consciousness_syncope`, `numbness_paresthesia`, `other_neurological`, `seizure_convulsion`, `speech_difficulty`, `vision_disturbance`, `weakness_paralysis`)
3. **Clinical Urgency Stratification** (3 classes: `Emergency`, `Urgent`, `Routine`)

**Scientific Guarantees**: 0 patient overlap across splits (`subject_id` isolated), 0 exact text overlap, 0 target-label leakage, pre-diagnostic bounding strictly enforced.

In [ ]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q transformers torch scikit-learn onnx onnxruntime matplotlib seaborn pandas

In [ ]:
# 2. Dataset Setup
import os, zipfile

os.makedirs('datasets/processed/neurologist', exist_ok=True)

# If dataset splits are uploaded as a zip archive
if os.path.exists('neurologist_splits.zip'):
    print("Unzipping neurologist_splits.zip...")
    with zipfile.ZipFile('neurologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/neurologist')
elif os.path.exists('/content/drive/MyDrive/neurologist_splits.zip'):
    print("Unzipping from Google Drive...")
    with zipfile.ZipFile('/content/drive/MyDrive/neurologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/neurologist')
elif not os.path.exists('datasets/processed/neurologist/train_split.json'):
    print("Please upload neurologist_splits.zip or train_split.json, val_split.json, test_split.json")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as z:
                z.extractall('datasets/processed/neurologist')

print("Datasets ready in datasets/processed/neurologist:")
for f in os.listdir('datasets/processed/neurologist'):
    if f.endswith('.json'):
        sz = os.path.getsize(os.path.join('datasets/processed/neurologist', f)) / (1024**2)
        print(f"  -> {f} ({sz:.2f} MB)")

In [ ]:
# 3. Reproducibility & Device Initialization
import time, json, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score, precision_recall_fscore_support
import matplotlib.pyplot as plt, seaborn as sns
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# 4. Taxonomy & Multi-Task Dataset
DISEASE_CLASSES = [
    'stroke_ischemic',
    'intracranial_hemorrhage',
    'epilepsy_seizures',
    'altered_mental_status_encephalopathy',
    'transient_ischemic_attack',
    'neuropathy_neurodegenerative'
]
COMPLAINT_CLASSES = [
    'altered_mental_status',
    'confusion_memory',
    'dizziness_vertigo',
    'headache_migraine',
    'loss_of_consciousness_syncope',
    'numbness_paresthesia',
    'other_neurological',
    'seizure_convulsion',
    'speech_difficulty',
    'vision_disturbance',
    'weakness_paralysis'
]
URGENCY_CLASSES = ['Emergency', 'Urgent', 'Routine']

COMPLAINT_MAP = {lbl: idx for idx, lbl in enumerate(COMPLAINT_CLASSES)}
URGENCY_MAP = {lbl: idx for idx, lbl in enumerate(URGENCY_CLASSES)}

class NeurologistMIMICDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=384):
        self.records = records
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.texts = [r.get('text', 'No clinical narrative.') for r in records]
        self.diseases = []
        for r in records:
            d_vec = [float(r.get('disease_labels', {}).get(cls, 0.0)) for cls in DISEASE_CLASSES]
            self.diseases.append(d_vec)
        
        self.complaints = [COMPLAINT_MAP.get(r.get('chief_complaint', 'other_neurological'), 6) for r in records]
        self.urgencies = [URGENCY_MAP.get(r.get('urgency', 'Urgent'), 1) for r in records]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'disease_label': torch.tensor(self.diseases[idx], dtype=torch.float),
            'complaint_label': torch.tensor(self.complaints[idx], dtype=torch.long),
            'urgency_label': torch.tensor(self.urgencies[idx], dtype=torch.long)
        }

In [ ]:
# 5. Multi-Task PubMedBERT Neural Architecture
class NeurologistMultiTaskClassifier(nn.Module):
    def __init__(self, model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract', num_diseases=6, num_complaints=11, num_urgencies=3, dropout_rate=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        self.dropout = nn.Dropout(dropout_rate)
        self.disease_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_diseases)
        )
        self.complaint_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_complaints)
        )
        self.urgency_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 4, num_urgencies)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(cls_rep)
        
        disease_logits = self.disease_head(pooled)
        complaint_logits = self.complaint_head(pooled)
        urgency_logits = self.urgency_head(pooled)
        
        return disease_logits, complaint_logits, urgency_logits

In [ ]:
# 6. Data Loading & Positive BCE Weight Computation
with open('datasets/processed/neurologist/train_split.json', 'r', encoding='utf-8') as f:
    train_records = json.load(f)
with open('datasets/processed/neurologist/val_split.json', 'r', encoding='utf-8') as f:
    val_records = json.load(f)
with open('datasets/processed/neurologist/test_split.json', 'r', encoding='utf-8') as f:
    test_records = json.load(f)

print(f"Cohort: Train={len(train_records)}, Val={len(val_records)}, Test={len(test_records)}")

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = NeurologistMIMICDataset(train_records, tokenizer, max_len=384)
val_ds = NeurologistMIMICDataset(val_records, tokenizer, max_len=384)
test_ds = NeurologistMIMICDataset(test_records, tokenizer, max_len=384)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

# Compute positive class weights for disease BCE
pos_counts = np.zeros(len(DISEASE_CLASSES))
for r in train_records:
    for i, c in enumerate(DISEASE_CLASSES):
        pos_counts[i] += r.get('disease_labels', {}).get(c, 0)
N_train = len(train_records)
pos_weights = np.sqrt((N_train - pos_counts) / np.clip(pos_counts, 1, None))
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float).to(DEVICE)
print("Positive BCE weights:", dict(zip(DISEASE_CLASSES, np.round(pos_weights, 3))))

In [ ]:
# 7. Loss, Optimizer & Training Loop
os.makedirs('experiments/neurologist_mimic_final', exist_ok=True)
model = NeurologistMultiTaskClassifier(MODEL_NAME).to(DEVICE)

disease_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
complaint_criterion = nn.CrossEntropyLoss()
urgency_criterion = nn.CrossEntropyLoss()

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        d_targets = batch['disease_label'].to(DEVICE)
        c_targets = batch['complaint_label'].to(DEVICE)
        u_targets = batch['urgency_label'].to(DEVICE)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            d_logits, c_logits, u_logits = model(input_ids, attention_mask)
            loss_d = disease_criterion(d_logits, d_targets)
            loss_c = complaint_criterion(c_logits, c_targets)
            loss_u = urgency_criterion(u_logits, u_targets)
            loss = 1.0 * loss_d + 0.7 * loss_c + 0.5 * loss_u
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()
    
    print(f"Epoch {epoch}/{EPOCHS} Loss: {running_loss/len(train_loader):.4f}")
    torch.save(model.state_dict(), 'experiments/neurologist_mimic_final/best_model.pt')

In [ ]:
# 8. Validation Threshold Optimization
model.eval()
val_preds, val_targets = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        d_logits, _, _ = model(input_ids, mask)
        val_preds.append(torch.sigmoid(d_logits).cpu().numpy())
        val_targets.append(batch['disease_label'].numpy())

val_preds = np.vstack(val_preds)
val_targets = np.vstack(val_targets)

best_thresholds = {}
for i, cls_name in enumerate(DISEASE_CLASSES):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.2, 0.85, 0.02):
        f = f1_score(val_targets[:, i], (val_preds[:, i] >= t).astype(int), zero_division=0)
        if f > best_f:
            best_f = f
            best_t = float(t)
    best_thresholds[cls_name] = best_t

print("Frozen Validation Thresholds:", json.dumps(best_thresholds, indent=2))
with open('experiments/neurologist_mimic_final/optimized_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)

In [ ]:
# 9. Authoritative Test Hold-Out Evaluation
test_d_preds, test_d_targets = [], []
test_c_preds, test_c_targets = [], []
test_u_preds, test_u_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        d_logits, c_logits, u_logits = model(input_ids, mask)
        test_d_preds.append(torch.sigmoid(d_logits).cpu().numpy())
        test_d_targets.append(batch['disease_label'].numpy())
        test_c_preds.append(torch.argmax(c_logits, dim=-1).cpu().numpy())
        test_c_targets.append(batch['complaint_label'].numpy())
        test_u_preds.append(torch.argmax(u_logits, dim=-1).cpu().numpy())
        test_u_targets.append(batch['urgency_label'].numpy())

test_d_preds = np.vstack(test_d_preds)
test_d_targets = np.vstack(test_d_targets)
test_c_preds = np.concatenate(test_c_preds)
test_c_targets = np.concatenate(test_c_targets)
test_u_preds = np.concatenate(test_u_preds)
test_u_targets = np.concatenate(test_u_targets)

test_d_binary = np.zeros_like(test_d_preds)
for i, cls_name in enumerate(DISEASE_CLASSES):
    test_d_binary[:, i] = (test_d_preds[:, i] >= best_thresholds[cls_name]).astype(int)

macro_f1 = f1_score(test_d_targets, test_d_binary, average='macro')
micro_f1 = f1_score(test_d_targets, test_d_binary, average='micro')
aurocs = [roc_auc_score(test_d_targets[:, i], test_d_preds[:, i]) for i in range(6)]
auprcs = [average_precision_score(test_d_targets[:, i], test_d_preds[:, i]) for i in range(6)]

print(f"\n--- TEST SET METRICS ---")
print(f"Disease Macro-AUROC: {np.mean(aurocs)*100:.2f}%")
print(f"Disease Macro-AUPRC: {np.mean(auprcs)*100:.2f}%")
print(f"Disease Macro-F1:    {macro_f1*100:.2f}%")
print(f"Disease Micro-F1:    {micro_f1*100:.2f}%")

In [ ]:
# 10. Production ONNX Export & Dynamic Quantization
os.makedirs('backend/models', exist_ok=True)
model.eval().to('cpu')
dummy_ids = torch.zeros((1, 384), dtype=torch.long)
dummy_mask = torch.zeros((1, 384), dtype=torch.long)

fp32_onnx_path = 'backend/models/neurologist_final.onnx'
torch.onnx.export(
    model,
    (dummy_ids, dummy_mask),
    fp32_onnx_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['disease_logits', 'complaint_logits', 'urgency_logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'},
        'attention_mask': {0: 'batch_size', 1: 'seq_len'},
        'disease_logits': {0: 'batch_size'},
        'complaint_logits': {0: 'batch_size'},
        'urgency_logits': {0: 'batch_size'}
    },
    opset_version=14
)

int8_onnx_path = 'backend/models/neurologist_final_quant.onnx'
quantize_dynamic(
    model_input=fp32_onnx_path,
    model_output=int8_onnx_path,
    weight_type=QuantType.QInt8
)

print(f"Exported FP32 ONNX: {os.path.getsize(fp32_onnx_path)/(1024**2):.2f} MB")
print(f"Exported INT8 ONNX: {os.path.getsize(int8_onnx_path)/(1024**2):.2f} MB")